# MalariAI — Phase 3: Pipeline B (Decoupled Framework)

**Kaysarul Anas Apurba · MalariAI Project**  
Pipeline B · Watershed + EfficientNet-B0 · Grad-CAM++

---

## 1 · Overview

Pipeline B addresses the limitations of standard object detectors (like Faster R-CNN) in dense, partially annotated malaria smears:

1.  **Stage 1: Watershed Segmentation** — Annotation-agnostic cell isolation.
2.  **Stage 2: EfficientNet-B0 Classifier** — Multi-class stage identification with Focal Loss.
3.  **Explainability: Grad-CAM++** — Spatial attention heatmaps for clinical validation.


## 2 · Setup & Imports

In [ ]:
import os, sys, json, random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from tqdm.auto import tqdm

# Add project root to sys.path for shared imports
# Notebook is at Phase3-PipelineB/On_mac/Phase3_PipelineB.ipynb
PROJECT_ROOT = Path(".").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "Phase1-EDA") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "Phase1-EDA"))

from shared.label_map import LABEL_TO_INT, INT_TO_LABEL, NUM_CLASSES, CLASS_COLOUR_RGB, SHORT_NAME
from dataset import MalariaDataset, MalariaCropDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 3 · Stage 1: Watershed Segmentation

Traditional CV to find *every* cell regardless of labels.

In [ ]:
def run_watershed(img_path):
    img = cv2.imread(str(img_path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # 1. Otsu thresholding
    ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # 2. Noise removal
    kernel = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # 3. Sure background
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    
    # 4. Sure foreground (distance transform)
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    ret, sure_fg = cv2.threshold(dist_transform, 0.4 * dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # 5. Markers
    ret, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    # 6. Watershed
    markers = cv2.watershed(img, markers)
    
    return img, markers

# Visualize on a random image
IMG_DIR = PROJECT_ROOT / "data/malaria/images"
all_images = list(IMG_DIR.glob("*.png"))
if all_images:
    test_img_path = random.choice(all_images)
    img, markers = run_watershed(test_img_path)
    
    plt.figure(figsize=(15, 7))
    plt.subplot(1, 2, 1)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis("off")
    
    # Colorized markers
    plt.subplot(1, 2, 2)
    plt.imshow(markers, cmap="jet")
    plt.title("Watershed Segments")
    plt.axis("off")
    plt.show()
else:
    print("No images found in data/malaria/images")

## 4 · Stage 2: EfficientNet-B0 Classifier

Training a robust classifier on isolated cell crops.

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch.nn as nn

def build_efficientnet(num_classes):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_efficientnet(NUM_CLASSES).to(device)
print(f"EfficientNet-B0 loaded with {NUM_CLASSES} classes")

## 5 · Explainability: Grad-CAM++

Providing spatial evidence for each classification.

In [ ]:
import torch.nn.functional as F

class GradCAMPlusPlus:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.target_layer.register_forward_hook(self.save_activations)
        self.target_layer.register_full_backward_hook(self.save_gradients)

    def save_activations(self, module, input, output): self.activations = output
    def save_gradients(self, module, grad_input, grad_output): self.gradients = grad_output[0]

    def generate(self, input_tensor, class_idx=None):
        self.model.eval()
        logit = self.model(input_tensor)
        if class_idx is None: class_idx = logit.argmax(dim=1).item()
        self.model.zero_grad()
        score = logit[0, class_idx]
        score.backward()

        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-10)
        return cam.detach().cpu().numpy()[0, 0], class_idx

def overlay_heatmap(img, heatmap, alpha=0.5):
    heatmap_resized = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img, 1 - alpha, heatmap_color, alpha, 0)

print("Grad-CAM++ Class defined.")

## 6 · End-to-End Pipeline Demo
Bringing it all together: Watershed → EfficientNet → Grad-CAM++

In [ ]:
from torchvision import transforms as T

def pipeline_demo(img_path, model, device):
    img, markers = run_watershed(img_path)
    
    # Extract crops for a few cells
    transform = T.Compose([
        T.ToPILImage(),
        T.Resize((64, 64)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    # Target layer is the last feature map
    target_layer = model.features[-1]
    gcam = GradCAMPlusPlus(model, target_layer)
    
    # Find a few segments to show
    # Label 1 is background, labels > 1 are objects
    labels = range(2, min(8, markers.max() + 1))
    if not labels:
        print("No segments found.")
        return
        
    plt.figure(figsize=(20, 5))
    
    for i, label in enumerate(labels):
        mask = np.zeros(img.shape[:2], dtype="uint8")
        mask[markers == label] = 255
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: continue
        
        x, y, w, h = cv2.boundingRect(cnts[0])
        # Crop with margin
        m = 5
        crop = img[max(0,y-m):y+h+m, max(0,x-m):x+w+m]
        
        input_tensor = transform(crop).unsqueeze(0).to(device)
        heatmap, pred_idx = gcam.generate(input_tensor)
        overlay = overlay_heatmap(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB), heatmap)
        
        plt.subplot(1, len(labels), i+1)
        plt.imshow(overlay)
        label_name = INT_TO_LABEL[pred_idx]
        plt.title(f"Pred: {SHORT_NAME[label_name]}")
        plt.axis("off")
    
    plt.show()

if all_images:
    pipeline_demo(random.choice(all_images), model, device)
